# Large Run Step 1: Ingest, Preprocess, and Context Tables

Purpose: build the standard Step 1 metadata and handoff tables without forcing waveform preprocessing or record coverage construction to run inside the notebook.

Outputs: `prepared_stations`, `prepared_events`, `event_station_records`, preprocessed waveform metadata, `record_coverage`, and context figures when requested.


## Setup
Purpose: load the active run config and define shared output, Slurm, and preview settings.

Outputs: printed repository/config/output paths and reusable helper variables.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import textwrap
import time

import pandas as pd

from spatial_vtk.config import SpatialVTKConfig
from spatial_vtk.config.outputs import resolve_output_path
from spatial_vtk.io import load_output_table, preview_output_table, write_output_table


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "spatial_vtk").exists():
            return candidate
    return start


repo_root = find_repo_root()
config_candidates = [
    repo_root / "runs" / "spatial_vtk_config.yaml",
    Path.cwd() / "spatial_vtk_config.yaml",
    Path.cwd() / "runs" / "spatial_vtk_config.yaml",
]
config_path = next((path for path in config_candidates if path.exists()), config_candidates[0])
cfg = SpatialVTKConfig.from_file(config_path).activate()

outputs_root = Path(cfg.path("outputs.root", create_parent=True) or (cfg.root_dir / "runs" / "outputs"))
tables_dir = Path(cfg.path("outputs.tables", create_parent=True) or (outputs_root / "tables"))
figures_dir = Path(cfg.path("outputs.figures", create_parent=True) or (outputs_root / "figures"))
slurm_dir = outputs_root / "slurm"
logs_dir = outputs_root / "logs"
for directory in (tables_dir, figures_dir, slurm_dir, logs_dir):
    directory.mkdir(parents=True, exist_ok=True)

RUN_LOCAL = os.environ.get("SVTK_RUN_LOCAL", "0") == "1"
SUBMIT_SLURM = os.environ.get("SVTK_SUBMIT_SLURM", "0") == "1"
OVERWRITE = os.environ.get("SVTK_OVERWRITE", "0") == "1"
QC_CHUNKSIZE = int(os.environ.get("SVTK_QC_CHUNKSIZE", "1000000"))
PREVIEW_ROWS = int(os.environ.get("SVTK_PREVIEW_ROWS", "5"))
PREPROCESS_CONTINUE_ON_ERROR = os.environ.get("SVTK_PREPROCESS_CONTINUE_ON_ERROR", "1") == "1"

print(f"repo_root: {repo_root}")
print(f"config_path: {config_path}")
print(f"outputs_root: {outputs_root}")
print(f"tables_dir: {tables_dir}")
print(f"figures_dir: {figures_dir}")
print(f"SUBMIT_SLURM={SUBMIT_SLURM} RUN_LOCAL={RUN_LOCAL} OVERWRITE={OVERWRITE}")
print(f"PREPROCESS_CONTINUE_ON_ERROR={PREPROCESS_CONTINUE_ON_ERROR}")


## Helpers
Purpose: define small skip, status, and Slurm-script helpers used by this notebook.

Outputs: helper functions only; no project files are written.


In [ ]:
def exists(path):
    return Path(path).exists()


def should_run(*paths):
    return OVERWRITE or not all(Path(path).exists() for path in paths)


def file_info(path):
    path = Path(path)
    if not path.exists():
        return {"path": str(path), "exists": False, "size_gb": None, "modified": None}
    return {
        "path": str(path),
        "exists": True,
        "size_gb": round(path.stat().st_size / 1024**3, 3),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }


def show_files(paths):
    display(pd.DataFrame([file_info(path) for path in paths]))


def submit_script(script_path, *, submit=SUBMIT_SLURM):
    script_path = Path(script_path)
    print(f"script: {script_path}")
    if submit:
        result = subprocess.run(["sbatch", str(script_path)], text=True, capture_output=True, check=False)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"sbatch failed with return code {result.returncode}")
    else:
        print(f"Set SVTK_SUBMIT_SLURM=1 and rerun this cell, or submit manually:")
        print(f"sbatch {shlex.quote(str(script_path))}")


def write_python_slurm_script(script_name, python_body, *, job_name, walltime="24:00:00", memory="32G", cpus=1):
    script_path = slurm_dir / script_name
    body = textwrap.dedent(python_body).strip()
    script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={logs_dir}/{job_name}_%j.out
#SBATCH --error={logs_dir}/{job_name}_%j.err
#SBATCH --time={walltime}
#SBATCH --mem={memory}
#SBATCH --cpus-per-task={cpus}

set -euo pipefail
cd {repo_root}
source /project2/jvidale_1700/miniforge3/etc/profile.d/conda.sh || true
conda activate spatial-vtk-py312 || true
python - <<'PYJOB'
{body}
PYJOB
"""
    script_path.write_text(script, encoding="utf-8")
    script_path.chmod(0o755)
    return script_path


def preview_table_path(path, n=PREVIEW_ROWS, columns=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    if path.suffix.lower() in {".parquet", ".pq"}:
        import pyarrow.parquet as pq
        parquet = pq.ParquetFile(path)
        available = parquet.schema.names
        selected = [column for column in (columns or available) if column in available]
        for batch in parquet.iter_batches(batch_size=max(int(n), 1), columns=selected):
            return batch.to_pandas().head(n)
        return pd.DataFrame(columns=selected)
    return pd.read_csv(path, nrows=n, usecols=columns)


## Resolve Standard Outputs
Purpose: identify the Step 1 files and show which ones already exist.

Outputs: a status table; no compute is started.


In [ ]:
prepared_stations_path = resolve_output_path("prepared_stations", kind="table", create_parent=True)
prepared_events_path = resolve_output_path("prepared_events", kind="table", create_parent=True)
event_station_path = resolve_output_path("event_station_records", kind="table", create_parent=True)
record_coverage_path = resolve_output_path("record_coverage", kind="table", create_parent=True)
preprocessed_root = outputs_root / "preprocessed_waveforms"
preprocessed_event_station_path = preprocessed_root / "metadata" / "event_station_records_preprocessed.csv"
preprocessed_trace_metadata_path = preprocessed_root / "metadata" / "trace_metadata_preprocessed.csv"
preprocessed_manifest_path = preprocessed_root / "metadata" / "preprocessing_manifest.csv"

show_files([
    prepared_stations_path,
    prepared_events_path,
    event_station_path,
    preprocessed_event_station_path,
    preprocessed_trace_metadata_path,
    preprocessed_manifest_path,
    record_coverage_path,
])


## Prepare Metadata
Purpose: standardize station/event metadata and build the event-station table. This is usually light enough to run in the notebook.

Outputs: `prepared_stations.csv`, `prepared_events.csv`, and `event_station_records.csv` if missing or `SVTK_OVERWRITE=1`.


In [ ]:
from spatial_vtk.io import prepare_station_metadata, prepare_event_metadata, prepare_event_station_table

if should_run(prepared_stations_path, prepared_events_path, event_station_path):
    stations = prepare_station_metadata()
    events = prepare_event_metadata()
    event_stations = prepare_event_station_table(station_metadata=stations, event_metadata=events)
    write_output_table("prepared_stations", stations)
    write_output_table("prepared_events", events)
    write_output_table("event_station_records", event_stations)
    print("Wrote prepared metadata tables.")
else:
    print("Prepared metadata tables already exist; skipping.")
    stations = load_output_table("prepared_stations")
    events = load_output_table("prepared_events")
    event_stations = load_output_table("event_station_records")

print(f"stations={len(stations):,} events={len(events):,} event_stations={len(event_stations):,}")


## Submit Waveform Preprocessing
Purpose: preprocess observed/synthetic waveforms in a Slurm job when the preprocessed metadata outputs are missing. Large runs commonly have partial observed/synthetic availability, so this cell continues through missing source files by default and drops only rows with no processed waveform path.

Outputs: `preprocessed_waveforms/metadata/event_station_records_preprocessed.csv`, `trace_metadata_preprocessed.csv`, and `preprocessing_manifest.csv`.


In [ ]:
if should_run(preprocessed_event_station_path, preprocessed_trace_metadata_path, preprocessed_manifest_path):
    script = write_python_slurm_script(
        "step01_preprocess_waveforms.slurm",
        f"""
        from spatial_vtk.config import SpatialVTKConfig
        from spatial_vtk.io import load_output_table
        from spatial_vtk.io.preprocessing import preprocess_waveform_files

        cfg = SpatialVTKConfig.from_file({str(config_path)!r}).activate()
        event_stations = load_output_table("event_station_records")
        result = preprocess_waveform_files(
            event_stations,
            config=cfg,
            verbose=True,
            overwrite={OVERWRITE!r},
            continue_on_error={PREPROCESS_CONTINUE_ON_ERROR!r},
        )
        print(f"event_station_records={{result.event_station_path}}")
        print(f"manifest={{result.manifest_path}}")
        print(f"trace_metadata={{result.trace_metadata_path}}")
        """,
        job_name="svtk-step01-preprocess",
        walltime="24:00:00",
        memory="32G",
        cpus=1,
    )
    submit_script(script)
else:
    print("Preprocessed waveform metadata already exists; skipping preprocessing submission.")


## Submit Record Coverage
Purpose: build `record_coverage.csv` from preprocessed trace metadata in a batch process, not inside the notebook.

Outputs: `record_coverage.csv` when trace metadata is available.


In [ ]:
if not preprocessed_trace_metadata_path.exists():
    print(f"Trace metadata is not ready yet: {preprocessed_trace_metadata_path}")
elif should_run(record_coverage_path):
    source_event_station_path = preprocessed_event_station_path if preprocessed_event_station_path.exists() else event_station_path
    script = write_python_slurm_script(
        "step01_record_coverage.slurm",
        f"""
        import pandas as pd
        from spatial_vtk.config import SpatialVTKConfig
        from spatial_vtk.io import read_table, write_output_table
        from spatial_vtk.visualize.context.figures import build_record_coverage_table_from_trace_metadata

        cfg = SpatialVTKConfig.from_file({str(config_path)!r}).activate()
        trace_metadata = read_table({str(preprocessed_trace_metadata_path)!r})
        event_stations = read_table({str(source_event_station_path)!r})
        record_coverage = build_record_coverage_table_from_trace_metadata(trace_metadata, event_station_df=event_stations)
        write_output_table("record_coverage", record_coverage)
        print(f"record_coverage rows={{len(record_coverage)}}")
        """,
        job_name="svtk-step01-coverage",
        walltime="12:00:00",
        memory="32G",
        cpus=1,
    )
    submit_script(script)
else:
    print("Record coverage table already exists; skipping.")


## Optional Context Figures
Purpose: make the Step 1 context figures only after the required tables exist. Keep this local because figures use small metadata tables.

Outputs: context, station/event coverage, and record coverage figures when missing.


In [ ]:
MAKE_FIGURES = os.environ.get("SVTK_MAKE_FIGURES", "0") == "1"
if not MAKE_FIGURES:
    print("Skipping figures. Set SVTK_MAKE_FIGURES=1 to render them.")
elif not record_coverage_path.exists():
    print("Skipping figures until record_coverage.csv exists.")
else:
    from spatial_vtk.visualize.context import (
        plot_event_coverage,
        plot_record_coverage,
        plot_station_coverage,
        plot_station_event_beachball_map,
        plot_station_event_context,
    )
    stations = load_output_table("prepared_stations")
    events = load_output_table("prepared_events")
    event_stations = load_output_table("event_station_records")
    record_coverage = load_output_table("record_coverage")
    plot_station_event_context(stations, events, add_basemap=True, showfig=False, savefig=True)
    plot_station_event_beachball_map(events, stations_df=stations, add_basemap=True, showfig=False, savefig=True)
    plot_station_coverage(event_stations, showfig=False, savefig=True)
    plot_event_coverage(event_stations, showfig=False, savefig=True)
    plot_record_coverage(record_coverage, showfig=False, savefig=True)
    print("Wrote Step 1 figures.")
